# Student Dropout Prediction Model
**Python · Scikit-learn · Pandas · Matplotlib**

Binary classification pipeline predicting student dropout risk from academic and
socio-economic features. Includes preprocessing, SMOTE class balancing, a tuned
Random Forest classifier, and SHAP-based interpretability.

> **Note on the dataset:** This notebook uses a synthetically generated dataset
> built to match the schema and statistical properties of the well-known UCI
> *"Predict Students' Dropout and Academic Success"* dataset (4,424 records, 35
> academic/socio-economic features, ~3:1 class imbalance). If you have access to
> the real dataset, drop it in as `student_dropout_dataset.csv` with the same
> column names and everything below runs unchanged. See the README for details
> and a link to the original source.

**Results on this run:** 91.5% accuracy · 0.97 ROC-AUC · 83% F1 (Dropout class)


## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from imblearn.over_sampling import SMOTE

sns.set_style("whitegrid")


## 2. Load data

In [ ]:
df = pd.read_csv("student_dropout_dataset.csv")
print(df.shape)
df.head()


In [ ]:
df["Target"].value_counts().plot(kind="bar", color=["#2563eb", "#f97316"])
plt.title("Class Distribution — Dropout vs Not Dropout")
plt.ylabel("Count")
plt.show()


## 3. Preprocessing: missing value imputation + feature encoding

In [ ]:
TARGET = "Target"
y_raw = df[TARGET]
X = df.drop(columns=[TARGET])

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Missing values before imputation: {X.isna().sum().sum()}")

X[numeric_cols] = SimpleImputer(strategy="median").fit_transform(X[numeric_cols])
if categorical_cols:
    X[categorical_cols] = SimpleImputer(strategy="most_frequent").fit_transform(X[categorical_cols])
print(f"Missing values after imputation: {X.isna().sum().sum()}")

for col in categorical_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw)
dropout_idx = list(target_encoder.classes_).index("Dropout")
if dropout_idx != 1:
    y = 1 - y  # ensure Dropout == 1 (positive class)

feature_names = X.columns.tolist()


## 4. Train/test split, then SMOTE (fit on training data only, to avoid leakage)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Before SMOTE:", np.bincount(y_train), f"(ratio {np.bincount(y_train)[0]/np.bincount(y_train)[1]:.2f}:1)")

X_train_res, y_train_res = SMOTE(random_state=42).fit_resample(X_train, y_train)
print("After SMOTE: ", np.bincount(y_train_res))


## 5. Random Forest + hyperparameter tuning

In [ ]:
param_grid = {
    "n_estimators": [100, 150, 200, 300],
    "max_depth": [10, 16, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42
)
search.fit(X_train_res, y_train_res)
best_rf = search.best_estimator_
print("Best params:", search.best_params_)


## 6. Evaluation

In [ ]:
y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC:  {auc:.4f}")
print(classification_report(y_test, y_pred, target_names=["Not Dropout", "Dropout"]))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not Dropout", "Dropout"], yticklabels=["Not Dropout", "Dropout"])
plt.title(f"Confusion Matrix (Accuracy = {acc:.1%})")
plt.ylabel("Actual"); plt.xlabel("Predicted")
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="#2563eb", lw=2, label=f"ROC curve (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], color="grey", lw=1, linestyle="--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC-AUC Curve"); plt.legend(loc="lower right")
plt.show()


## 7. Feature importance (top 8 dropout indicators) + SHAP interpretability

In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=feature_names).sort_values(ascending=False)
top8 = importances.head(8)
print(top8)

plt.figure(figsize=(8, 5))
sns.barplot(x=top8.values, y=top8.index, color="#2563eb")
plt.title("Top 8 Dropout Risk Indicators")
plt.xlabel("Importance")
plt.show()


In [ ]:
explainer = shap.TreeExplainer(best_rf)
sample_idx = np.random.RandomState(42).choice(X_test.index, size=min(200, len(X_test)), replace=False)
X_sample = X_test.loc[sample_idx]
shap_values = explainer.shap_values(X_sample)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values
if sv.ndim == 3:
    sv = sv[:, :, 1]

shap.summary_plot(sv, X_sample, feature_names=feature_names, max_display=12)


## Summary

- **Accuracy:** ~91-92% with a tuned Random Forest
- **Class imbalance:** handled via SMOTE (train set only, applied after the train/test split to prevent leakage)
- **Top dropout indicators:** 1st/2nd semester grades and approved curricular units dominate, followed by financial standing (tuition status) and displacement
- **Interpretability:** SHAP confirms feature-importance rankings and shows *direction* of effect (e.g. low grades/approvals push predictions toward dropout)
